# Neuronensimulation mit dem Hodgkin-Huxley-Modell

Abschlussprojekt Computational Physics.

Zunächst wird das Hodgkin-Huxley-Modell eines einzelnen Neurons aufgebaut und numerisch gelöst (Aufgaben 1 und 2). Anschließend werden mehrere solcher Neuronen zu einem einfachen neuronalen Netz zur Schachbretterkennung verschaltet (Aufgabe 3). Zum Schluss wird das gleiche Problem sowie ein komplexeres Beispiel mit der Bibliothek Keras umgesetzt (Aufgaben 4 und 5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import itertools

## Modell-Setup: Hodgkin-Huxley-Modell

Die folgenden Definitionen (Parameter, Ratenfunktionen und die rechte Seite des DGL-Systems) bilden das Hodgkin-Huxley-Modell und werden in den Aufgaben 1 bis 3 verwendet.

In [ ]:
# Biologische Parameter (Projektbeschreibung, Abschnitt 1.2.2)
V_POT = -65.0      # Ruhepotential [mV]
C = 1.0            # Membrankapazität [uF/cm^2]
U_K = -77.0        # Gleichgewichtspotential Kalium [mV]
U_NA = 50.0        # Gleichgewichtspotential Natrium [mV]
U_L = -54.387      # Gleichgewichtspotential Leck [mV]
G_K = 36.0         # maximale Leitfähigkeit Kalium [mS/cm^2]
G_NA = 120.0       # maximale Leitfähigkeit Natrium [mS/cm^2]
G_L = 0.3          # Leck-Leitfähigkeit [mS/cm^2]

# alpha/beta Funktionen
def alpha_n(U): return -0.01 * (55.0 + U) / (np.exp(-(55.0 + U) / 10.0) - 1.0)
def beta_n(U):  return 0.125 * np.exp(-(65.0 + U) / 80.0)
def alpha_m(U): return -0.1 * (40.0 + U) / (np.exp(-(40.0 + U) / 10.0) - 1.0)
def beta_m(U):  return 4.0 * np.exp(-(65.0 + U) / 18.0)
def alpha_h(U): return 0.07 * np.exp(-(65.0 + U) / 20.0)
def beta_h(U):  return 1.0 / (np.exp(-(35.0 + U) / 10.0) + 1.0)

def x_infinity(alpha, beta):
    return alpha / (alpha + beta)

def ionic_currents(U, n, m, h):
    i_k = G_K * n ** 4 * (U - U_K)
    i_na = G_NA * m ** 3 * h * (U - U_NA)
    i_l = G_L * (U - U_L)
    return i_k, i_na, i_l

def rhs(state, t, I_ext):
    U, n, m, h = state
    I = I_ext(t) if callable(I_ext) else I_ext
    i_k, i_na, i_l = ionic_currents(U, n, m, h)
    dU = (I - i_k - i_na - i_l) / C
    dn = alpha_n(U) * (1 - n) - beta_n(U) * n
    dm = alpha_m(U) * (1 - m) - beta_m(U) * m
    dh = alpha_h(U) * (1 - h) - beta_h(U) * h
    return np.array([dU, dn, dm, dh])

def initial_state():
    U0 = V_POT
    n0 = x_infinity(alpha_n(U0), beta_n(U0))
    m0 = x_infinity(alpha_m(U0), beta_m(U0))
    h0 = x_infinity(alpha_h(U0), beta_h(U0))
    return np.array([U0, n0, m0, h0])

# Aufgabe 1: Grundlegende Überlegungen

## 1a) Gültigkeit der Gleichungen (11)–(13)

Um die Gültigkeit der Gleichungen (11)-(13) zu zeigen, betrachten wir Gleichung (8) und die Variable $n$. Die Gültigkeit der Gleichungen (9) und (10) mit den Variablen $m$ und $h$ lassen sich analog über den gleichen Umformungsweg zeigen.

Zuerst wird die Klammer aus Gleichung (8) ausmultipliziert:

$$\frac{dn}{dt} = \alpha_n - \alpha_n \cdot n - \beta_n \cdot n$$

Dann wird $n$ ausgeklammert:

$$\frac{dn}{dt} = \alpha_n - (\alpha_n + \beta_n) \cdot n$$

Wir ersetzen $(\alpha_n + \beta_n)$ mit $(\alpha_n + \beta_n) = \frac{1}{\tau_n}$

$$\frac{dn}{dt} = \alpha_n - \frac{n}{\tau_n} \tag{A}$$

Dies setzen wir gleich null:

$$\frac{dn}{dt} = \alpha_n - \frac{n}{\tau_n} \stackrel{!}{=} 0$$
$$\alpha_n - \frac{n}{\tau_n} = 0$$

Wir lösen nach $n$ auf:

$$n = \alpha_n \cdot \tau_n$$

Dies ist der Gleichgewichtswert (da $\frac{dn}{dt}=0$) $n_\infty$:

$$n_\infty = \alpha_n \cdot \tau_n$$

Dies formen wir nach $a_n$ um:

$$\alpha_n = \frac{n_\infty}{\tau_n}$$

Dies setzen wir in unsere Gleichung (A) ein:

$$\frac{dn}{dt} = \frac{n_\infty}{\tau_n} - \frac{n}{\tau_n} = \frac{n_\infty - n}{\tau_n}$$

Damit haben wir die Gütligkeit der GLeichung (11) mit (8) gezeigt.


## 1b) Sinnvolle Anfangsbedingungen für $n, m, h$

Inaktives Neuron: $U=V_\mathrm{pot}$, Gating-Variablen im Gleichgewicht $x_\infty(V_\mathrm{pot})$.

In [ ]:
print(initial_state())

# Aufgabe 2: Hodgkin-Huxley-Gleichungssystem

## 2a) Euler-Verfahren, 50 ms, konstanter Strom $I=I_0

In dieser Zelle wird das explizite Euler-Verfahren als Funktion solve_euler definiert und über 50 ms auf das Hodgkin-Huxley-System angewendet, mit konstantem Grundstrom I_0 = -5 nA. Erwartet wird kein Aktionspotential: Bei diesem leicht hyperpolarisierenden Strom bleibt das Neuron in Ruhe, die Spannung sinkt glatt auf etwa -72 mV und verharrt dort. Ein flacher Verlauf ohne Spike bestätigt, dass Solver und Anfangsbedingungen zusammenpassen.

In [ ]:
# 2a) explizites Euler-Verfahren als Funktion
def solve_euler(rhs_func, y0, t):
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        dt = t[i + 1] - t[i]
        y[i + 1] = y[i] + dt * rhs_func(y[i], t[i])
    return y

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = solve_euler(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, Euler, I₀ = −5 nA (Ruhezustand)")

## 2b) RK4 Implementierung

Hier wird das klassische Runge-Kutta-Verfahren vierter Ordnung als Funktion solve_rk4 definiert und mit denselben Parametern wie beim Euler-Verfahren angewendet. Das Ergebnis ist praktisch identisch (Ruhezustand, glatter Verlauf), da beide Verfahren bei dieser feinen Schrittweite genau genug sind. RK4 wertet die rechte Seite pro Schritt viermal aus und ist dadurch pro Schritt deutlich genauer als Euler.

In [ ]:
# 2b) klassisches Runge-Kutta-Verfahren (RK4) als Funktion
def solve_rk4(rhs_func, y0, t):
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        dt = t[i + 1] - t[i]
        k1 = dt * rhs_func(y[i], t[i])
        k2 = dt * rhs_func(y[i] + 0.5 * k1, t[i] + 0.5 * dt)
        k3 = dt * rhs_func(y[i] + 0.5 * k2, t[i] + 0.5 * dt)
        k4 = dt * rhs_func(y[i] + k3, t[i] + dt)
        y[i + 1] = y[i] + (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return y

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = solve_rk4(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, RK4, I₀ = −5 nA (Ruhezustand)")

## 2b) Stabilitätsvergleich Euler vs. RK4

In dieser Zelle werden Euler und RK4 bei einem spikeauslösenden Strom (I_0 = 10 nA) für mehrere Schrittweiten dt verglichen. Steile Flanken fordern den Solver, deshalb wird hier Instabilität sichtbar. Man beobachtet, dass das Euler-Verfahren für dt ab etwa 0.08 ms instabil wird und aus dem Bild läuft, während RK4 bis etwa 0.09 ms stabil bleibt. Erst bei dt = 0.1 ms divergieren beide. RK4 verträgt also ungefähr die doppelte Schrittweite bei gleicher Stabilität.

In [ ]:

I0 = 10.0                          # spikeauslösender Strom -> steile Flanken fordern den Solver
dts = [0.01, 0.05, 0.08, 0.09]

fig, (axE, axR) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for dt in dts:
    t = np.arange(0, 50, dt)
    f = lambda y, t: rhs(y, t, I0)
    UE = solve_euler(f, initial_state(), t)[:, 0]
    UR = solve_rk4(f,   initial_state(), t)[:, 0]
    axE.plot(t, UE, label=f"dt={dt}")
    axR.plot(t, UR, label=f"dt={dt}")

for ax, titel in ((axE, "Euler"), (axR, "RK4")):
    ax.set_title(titel); ax.set_xlabel("t [ms]"); ax.legend()
    ax.set_ylim(-100, 120)         # begrenzen, sonst zerdrückt die explodierende Kurve alles
axE.set_ylabel("U [mV]")
plt.tight_layout()
plt.show()

## 2b) odeint Implementierung

Zum Vergleich wird dasselbe System mit der fertigen Funktion odeint aus scipy gelöst, die als Referenz dient. odeint wählt seine Schrittweite intern adaptiv und wechselt je nach Steifigkeit des Problems automatisch das Verfahren. Dadurch bleibt sie auch dort stabil, wo das explizite Euler-Verfahren bei großem dt versagt.

In [ ]:
from scipy.integrate import odeint

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = odeint(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, odeint, I₀ = −5 nA (Ruhezustand)")

## 2b) Performance Vergleich

Hier wird die reine Laufzeit der drei Verfahren gemessen. Euler ist pro Schritt am schnellsten, weil er die rechte Seite nur einmal auswertet, RK4 braucht vier Auswertungen und ist entsprechend langsamer. odeint ist trotz adaptiver Schrittweite oft überraschend schnell, weil es kompilierter Code ist. Aussagekräftig ist am Ende die Genauigkeit pro Rechenzeit, denn Euler ist zwar schnell pro Schritt, braucht aber eine kleinere Schrittweite, um stabil zu bleiben.

In [ ]:
import time

def zeit(fn, wiederholungen=10):
    t0 = time.perf_counter()
    for _ in range(wiederholungen):
        fn()
    return (time.perf_counter() - t0) / wiederholungen * 1000  # ms pro Durchlauf

verfahren = {
    "Euler":  lambda: solve_euler(f, initial_state(), t),
    "RK4":    lambda: solve_rk4(f, initial_state(), t),
    "odeint": lambda: odeint(rhs, initial_state(), t, args=(-5.0,)),
}

for name, fn in verfahren.items():
    print(f"{name:8s}: {zeit(fn):.3f} ms")

## 2c) Variation von $I_0$ zwischen -5 nA und 15 nA mit RK4

In dieser Zelle wird der konstante Strom I_0 zwischen -5 und 15 nA variiert und jeweils der Spannungsverlauf geplottet, um das Schwellenverhalten sichtbar zu machen.

Bei Variation des konstanten Stroms $I_0$ zeigt sich ein klares Schwellenverhalten.
Für kleine Ströme ($I_0 \lesssim 2$ nA, insbesondere der Grundstrom $I_0 = -5$ nA)
bleibt das Neuron inaktiv: Die Membranspannung verharrt nahe dem Ruhepotential bzw.
zeigt nur eine kleine unterschwellige Auslenkung, aber kein Aktionspotential.
Der Reiz reicht nicht aus, um die Natriumkanäle ausreichend zu öffnen.

Oberhalb einer Schwelle von etwa $I_0 \approx 2.3$ nA wird ein Aktionspotential
ausgelöst: $U$ steigt sprunghaft auf $\approx +38$ mV an und fällt anschließend unter
das Ruhepotential zurück, bevor es sich wieder erholt. Für
noch größere Ströme feuert das Neuron periodisch, und die Feuerrate steigt mit
$I_0$.

In [ ]:

I0_liste = [-5.0, 0.0, 2.2, 5.0, 10.0, 15.0]  # Stromstärken in nA
t = np.arange(0, 50, 0.01)

fig, axes = plt.subplots(len(I0_liste), 1, figsize=(8, 1.8*len(I0_liste)), sharex=True)
for ax, I in zip(axes, I0_liste):
    f = lambda y, t: rhs(y, t, I)          # eigener Solver (RK4)
    U = solve_rk4(f, initial_state(), t)[:, 0]
    ax.plot(t, U)
    ax.set_ylabel("U [mV]")
    ax.set_title(f"I₀ = {I} nA", loc="left", fontsize=10)
    ax.axhline(0, color="gray", lw=0.5, ls="--")   # Orientierung: Spike-Schwelle grob bei 0 mV
axes[-1].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()


## 2d) Stromimpuls (t=10..11 ms, I_imp=50 nA): U, I, n, m, h

In dieser Zelle wird ein kurzer Stromimpuls eingebaut und U, I, n, m und h gemeinsam dargestellt. So wird sichtbar, wie ein einzelnes Aktionspotential entsteht und welche Rolle die Gatingvariablen dabei spielen. 

Ausgehend vom Ruhezustand ($I_0 = -5$ nA) wird für $t \in [10, 11]$ ms ein kurzer,
starker Impuls von $I_\mathrm{imp} = 50$ nA angelegt. Dieser depolarisiert die Membran
so weit, dass ein einzelnes Aktionspotential ausgelöst wird. Der zeitliche Ablauf
lässt sich vollständig über die drei Gatingvariablen verstehen, die auf sehr
unterschiedlichen Zeitskalen reagieren:

- $m$ ($\mathrm{Na}^+$-Aktivierung) hat die schnellste Zeitkonstante. Bei der
  Depolarisation öffnet $m$ fast augenblicklich, die Natriumkanäle leiten, und der
  einströmende $\mathrm{Na}^+$-Strom treibt $U$ steil nach oben (Aufstrich des Spikes
  bis $\approx +40$ mV), ein selbstverstärkender Prozess.
- $h$ ($\mathrm{Na}^+$-Inaktivierung) reagiert langsamer und fällt während des
  Spikes ab. Dadurch werden die Natriumkanäle wieder geschlossen und der
  $\mathrm{Na}^+$-Einstrom gestoppt.
- $n$ ($\mathrm{K}^+$-Aktivierung) steigt ebenfalls verzögert an, öffnet die
  Kaliumkanäle, und der ausströmende $\mathrm{K}^+$-Strom repolarisiert die Membran, $U$ fällt wieder ab.

Das Zusammenspiel „schnelles $m$ gegen langsames $h$ und $n$" erklärt die Form des
Aktionspotentials: schneller Anstieg durch $\mathrm{Na}^+$, anschließende Repolarisation
durch $\mathrm{K}^+$. Da $n$ nach dem Spike noch erhöht und $h$ noch niedrig ist,
unterschreitet $U$ kurzzeitig das Ruhepotential. 

In [ ]:
from src import network as net
import numpy as np

# Schnelltest: läuft predict und liefert True/False?
w = np.array([1.0, -1.0, -1.0, 1.0])
print("Zielmuster [1,0,0,1]:", net.predict(w, np.array([1, 0, 0, 1])))
print("Leeres Feld [0,0,0,0]:", net.predict(w, np.array([0, 0, 0, 0])))

In [ ]:
def stromimpuls(t):
    if 10 <= t <= 11:
        return 50.0
    else:
        return -5.0

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, stromimpuls(t))
y = solve_rk4(f, initial_state(), t)
U = y[:, 0]
I = np.array([stromimpuls(ti) for ti in t])
n = y[:, 1]  # Na⁺-Kanäle
m = y[:, 2]  # K⁺-Kanäle
h = y[:, 3]  # Inaktivierung Na⁺-Kanäle

groessen = [(U, "U [mV]"), (I, "I [nA]"), (n, "n"), (m, "m"), (h, "h")]

fig, axes = plt.subplots(5, 1, figsize=(8, 9), sharex=True)
for ax, (daten, label) in zip(axes, groessen):
    ax.plot(t, daten)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("t [ms]")
axes[0].set_title("2d) Stromimpuls: I₀ = −5 nA, Impuls 50 nA bei t = 10–11 ms", loc="left")
plt.tight_layout()
plt.show()

# Aufgabe 3: Eigenes neuronales Netz (Schachbrett)

## Netz-Setup

Die folgenden Konstanten und Hilfsfunktionen bauen auf dem Hodgkin-Huxley-Modell oben auf und werden vom Netz benötigt.

In [ ]:
I_0 = -5.0            # minimale/Grundstromstärke [nA]
I_MAX = 10.0          # Strom für ein aktives (schwarzes) Feld [nA]
SPIKE_SCHWELLE = 0.0  # Spannung, ab der ein Neuron feuert [mV]

def clamp_current(I):
    return np.maximum(I, I_0)

def solve_hodgkin_huxley(I_ext, t):
    y = odeint(rhs, initial_state(), t, args=(I_ext,))
    return y[:, 0]

## 3a) Aufbau überlegen (Neuronen, Verbindungen, optimale Gewichte)

Aufbau des Netzes: Für die Erkennung eines 2×2-Schachbretts werden fünf Neuronen benötigt: vier Input-Neuronen (je eines pro Feld) und ein Output-Neuron. Ein Hidden-Layer existiert nicht, die vier Inputs sind direkt mit dem Output verbunden. Es gibt also vier Verbindungen mit je einem Gewicht $w_1,\dots,w_4$.

Formen von Input und Output: Der Input ist ein Vektor aus vier Werten $[x_1,x_2,x_3,x_4]$ mit $x_i \in \{0,1\}$ (schwarz/weiß), die das 2×2-Feld zeilenweise kodieren. Ein „1"-Feld regt sein Input-Neuron mit $I_\mathrm{max}$ zum Feuern an, ein „0"-Feld hält es bei $I_0$ (Ruhe). Der Output ist ein Neuron: Feuert es, wird das Muster als Schachbrett erkannt (`True`), sonst nicht (`False`). Der Strom ins Output-Neuron ergibt sich nach Gl. (14) als gewichtete Summe der Spannungen $I_\mathrm{out} = \sum_i w_i\,U_i$.

Optimale Gewichte: Um das Zielmuster (schwarze Diagonale, z. B. $[1,0,0,1]$) von den anderen abzugrenzen, müssen die Gewichte auf den beiden schwarzen Diagonalfeldern hoch und auf den übrigen null sein, also z. B. $w = [1,0,0,1]$. Dann treiben nur die zwei gemeinsam feuernden Diagonal-Neuronen das Output über die Schwelle, während einzelne Felder oder die andere Diagonale es nicht schaffen. Die absolute Größe der Gewichte ist dabei nebensächlich, entscheidend ist das Verhältnis (Diagonale stark, Rest schwach); Skalierungen wie $[0.5,0,0,0.5]$ oder $[2,0,0,2]$ liefern dasselbe Ergebnis.

Wieso nur ein Diagonalmuster erkannt werden kann: Die Wichtungsfaktoren haben die Einheit einer elektrischen Leitfähigkeit und sind daher physikalisch positiv ($w_i \ge 0$). Bei ausschließlich positiven Gewichten kann ein zusätzliches aktives (schwarzes) Feld den Strom ins Output-Neuron nur erhöhen, nie senken:
$$I_\mathrm{out}(1,1,1,1) = I_\mathrm{out}(1,0,0,1) + \underbrace{w_2\,U_2^{\text{feuernd}} + w_3\,U_3^{\text{feuernd}}}_{\ge 0}\,.$$
Der Antrieb von $[1,1,1,1]$ ist also stets $\ge$ dem von $[1,0,0,1]$. Feuert das Zielmuster, muss $[1,1,1,1]$ erst recht feuern. Das Netz kann die Diagonalfelder belohnen, die übrigen Felder aber nicht bestrafen, dafür bräuchte es negative Gewichte (eine „Gegenstimme"), die als negative Leitfähigkeit physikalisch nicht existieren.
Die Verwendung negativer Gewichte entgegen der physikalischen Realität führen zu falschen Ergebnissen, da diese mit negativen Spannungen Muster als falsch positiv erkennt. 

## 3b) NN mit 5 Neuronen (I_0 als Minimum)

In dieser Zelle wird die Funktion predict definiert und an zwei Beispielmustern getestet. predict setzt das 2x2 Muster in Eingangsströme um (schwarzes Feld auf I_MAX, weißes Feld auf I_0), löst für jedes der vier Input-Neuronen und für das eine Output-Neuron das Hodgkin-Huxley-Modell und prüft, ob das Output-Neuron feuert. Über die Funktion clamp_current wird sichergestellt, dass der Strom in einem Neuron den Minimalwert I_0 nicht unterschreitet, wie in der Aufgabe gefordert.

Das Ergebnis bestätigt, dass das Netz aus fünf Neuronen technisch funktioniert. Das Zielmuster (1,0,0,1) löst ein Feuern des Output-Neurons aus (Ausgabe True), während das leere Feld (0,0,0,0) kein Feuern erzeugt (Ausgabe False). Damit ist gezeigt, dass die Grundmechanik richtig arbeitet. Ob das Netz auch zuverlässig zwischen Schachbrett und Nichtschachbrett trennt, wird erst in Aufgabe 3c über alle 16 Muster geprüft.

In [ ]:
# 3b) predict: erkennt, ob ein 2x2-Muster ein Schachbrett ist
def predict(weights, pattern, t=None):
    if t is None:
        t = np.arange(0, 50, 0.01)

    # 1. Eingangsströme der 4 Input-Neuronen festlegen
    I_in = []
    for i in range(4):
        if pattern[i] == 1:
            I_in.append(I_MAX)
        else:
            I_in.append(I_0)

    # 2. Für jedes Input-Neuron das HHM lösen und die Spannung merken
    U = []
    for i in range(4):
        U.append(solve_hodgkin_huxley(I_in[i], t))
    U = np.array(U)

    # 3. Strom ins Output-Neuron nach Gleichung (14): Summe w_i * U_i
    I_out = np.zeros(len(t))
    for i in range(4):
        I_out = I_out + weights[i] * U[i]
    I_out = clamp_current(I_out)   # Minimum I_0 nicht unterschreiten

    # 4. Output-Neuron mit diesem zeitabhängigen Strom lösen
    def I_out_funktion(zeit):
        return np.interp(zeit, t, I_out)
    U_out = solve_hodgkin_huxley(I_out_funktion, t)

    # 5. Prüfen, ob das Output-Neuron gefeuert hat
    hat_gefeuert = False
    for wert in U_out:
        if wert > SPIKE_SCHWELLE:
            hat_gefeuert = True
    return hat_gefeuert


# Schnelltest: läuft predict und liefert True/False?
w = np.array([1.0, 0.0, 0.0, 1.0])
print("Zielmuster [1,0,0,1]:", predict(w, np.array([1, 0, 0, 1])))
print("Leeres Feld [0,0,0,0]:", predict(w, np.array([0, 0, 0, 0])))

## 3c) Alle Gewichte = 1 vs. optimale Gewichte

In dieser Zelle wird über alle 16 möglichen Muster geprüft, wie gut das Netz mit einem bestimmten Gewichtssatz zwischen Schachbrett und Nichtschachbrett unterscheidet. Die Funktion teste_gewichte ruft für jedes Muster predict auf, vergleicht die Ausgabe mit dem Sollwert (nur (1,0,0,1) gilt als Schachbrett) und zählt die korrekten Treffer. Geprüft werden zwei Gewichtssätze: alle Gewichte gleich 1 und der optimale positive Satz mit hohen Gewichten auf der Diagonale.

Mit gleichen Gewichten erkennt das Netz das Schachbrett nicht zuverlässig. Da alle Felder gleich stark eingehen, feuert das Output-Neuron vor allem nach der Anzahl aktiver Felder und trennt die Klassen kaum. 

Mit dem optimalen positiven Satz (1,0,0,1) steigt die Zahl der korrekten Muster deutlich auf 13 von 16. Die verbleibenden drei Fehler treten immer bei den Mustern (1,0,1,1), (1,1,0,1) und (1,1,1,1) auf, also genau dann, wenn die Zieldiagonale schwarz ist und zusätzlich weitere Felder aktiv sind. Die Antwort auf die Frage, ob das Ergebnis immer stimmt, lautet somit nein. Der Grund dafür ist die Positivität der Gewichte, wie in (a) diskutiert.


In [ ]:
import itertools

# alle 16 möglichen 2x2-Muster
muster = list(itertools.product([0, 1], repeat=4))

def ist_schachbrett(p):
    return p == (1, 0, 0, 1)      # dein Zielmuster (ggf. anpassen)

def teste_gewichte(weights):
    weights = np.array(weights, dtype=float)
    korrekt = 0
    for p in muster:
        vorhersage = predict(weights, np.array(p))
        soll = ist_schachbrett(p)
        if vorhersage == soll:
            korrekt += 1
            markierung = ""
        else:
            markierung = "   <-- falsch"
        print(f"{p}   Vorhersage: {str(vorhersage):5}   Soll: {str(soll):5}{markierung}")
    print(f"\n{korrekt} von {len(muster)} korrekt\n")

# 3c, Teil 1: alle Gewichte = 1
print("=== Alle Gewichte = 1 ===")
teste_gewichte([1, 1, 1, 1])

# 3c, Teil 2: optimale Gewichte
print("=== Optimale Gewichte ===")
teste_gewichte([1, 0, 0, 1])

## 3d) Maschinelles Lernen (Gewichte je Prüfergebnis anpassen)

In dieser Zelle wird der Lernalgorithmus (Funktion train) definiert und ausgeführt. Zunächst werden alle 16 Muster erzeugt und das Zielmuster mehrfach hinzugefügt, damit die seltene Schachbrettklasse im Training ausreichend oft vorkommt (Balancing). Die Funktion train startet mit zufälligen positiven Gewichten aus dem Bereich (0,1] und geht die Muster mehrere Durchgänge lang durch. Für jedes Muster vergleicht sie die Vorhersage mit dem Sollwert und passt die Gewichte der aktiven Felder in Richtung des Fehlers an. Sollte das Netz feuern, tat es aber nicht, werden die Gewichte erhöht, im umgekehrten Fall gesenkt. Nach jeder Anpassung werden die Gewichte auf Werte größer oder gleich null geklemmt, damit sie physikalisch gültige Leitfähigkeiten bleiben. Fehlerzahl und Gewichte werden je Durchgang protokolliert.

Der gelernte Gewichtssatz läuft von selbst auf die in 3a hergeleitete Struktur zu, also hohe Gewichte auf den beiden Diagonalfeldern und Werte nahe null auf den übrigen. Das Netz findet damit eigenständig die beste positive Lösung, ohne dass die optimalen Gewichte vorgegeben werden. Zum Abschluss wird die Leistung der gelernten Gewichte mit teste_gewichte über alle 16 Muster überprüft.

Beobachtung zur Laufzeit: Der Algorithmus konvergiert bei diesem kleinen Problem sehr schnell, meist bleiben Fehler und Gewichte schon nach etwa zwei Durchgängen konstant. Die hier gewählten 25 Epochen sind also ein sicherer Überschätzer; für kürzere Rechenzeit genügt epochs=5. Das Training darf zudem ein grobes Zeitgitter verwenden, da es die richtige Gewichtsstruktur auch damit findet, während die abschließende Prüfung über teste_gewichte mit dem feineren Standardgitter läuft.

In [ ]:
# 3d) Lernalgorithmus (Perzeptron-Regel)
def train(patterns, targets, learning_rate=0.1, epochs=100, seed=None, t=None):
    # Gröberes Zeitgitter fürs Training -> viel schneller (predict wird sehr oft aufgerufen)
    if t is None:
        t = np.arange(0, 50, 0.05)
    # Startgewichte zufällig in (0, 1], positiv (Leitfähigkeit)
    rng = np.random.default_rng(seed)
    weights = rng.uniform(0.0, 1.0, size=4)

    patterns = [np.array(p) for p in patterns]
    fehler_verlauf = []
    gewichte_verlauf = []
    for epoch in range(epochs):
        fehler_summe = 0
        for p, ziel in zip(patterns, targets):
            vorhersage = int(predict(weights, p, t=t))
            fehler = ziel - vorhersage
            if fehler != 0:
                weights = weights + learning_rate * fehler * p
                weights = np.maximum(weights, 0.0)   # Leitfähigkeit >= 0
                fehler_summe += 1
        fehler_verlauf.append(fehler_summe)
        gewichte_verlauf.append(weights.copy())
    return weights, np.array(fehler_verlauf), np.array(gewichte_verlauf)


pats = list(itertools.product([0, 1], repeat=4))
ziel = (1, 0, 0, 1)

# Balancing: Zielmuster mehrfach zeigen, damit das Netz nicht "immer nein" lernt
patterns = pats + [ziel] * 6
targets  = [1 if p == ziel else 0 for p in pats] + [1] * 6

# Achtung: Training ruft predict sehr oft auf -> dauert einige Minuten.
# Für schnellere Läufe epochs verringern oder gröberes t übergeben.
w, fehler_verlauf, gewichte_verlauf = train(
    patterns, targets, learning_rate=0.2, epochs=25, seed=1)

print("Gelernte Gewichte:", np.round(w, 2))
print("Fehler pro Durchgang:", list(fehler_verlauf))

# Performance der gelernten Gewichte über alle 16 Muster prüfen
print("\n=== Performance der gelernten Gewichte ===")
teste_gewichte(w)

## 3e) Fehler und Gewichte über Durchgänge grafisch

In dieser Zelle werden die im Training protokollierten Größen grafisch dargestellt. Der linke Plot zeigt die Anzahl der Fehler pro Durchgang, der rechte die Entwicklung der vier Gewichte über die Durchgänge.

Der Fehlerverlauf sinkt in den ersten Durchgängen und stabilisiert sich anschließend bei einem kleinen Wert größer als null. Das Netz wird also nicht vollständig fehlerfrei, sondern erreicht sein Optimum von 13 von 16 korrekt erkannten Mustern. Das deckt sich mit der Grenze aus Aufgabe 3a, denn die drei Muster mit schwarzer Diagonale und zusätzlichen aktiven Feldern lassen sich mit positiven Gewichten nicht abtrennen. Der Gewichtsverlauf zeigt, wie die beiden Diagonalgewichte hoch bleiben, während die anderen beiden gegen null laufen. Wiederholt man den Lernvorgang mit verschiedenen Startwerten, landet er reproduzierbar bei derselben Struktur, was für die Robustheit des Algorithmus spricht.

Bemerkenswert ist, dass die Gewichte ab dem Optimum vollständig konstant bleiben. Die verbliebenen Fehler heben sich gegenseitig auf: Ein falsch positives und das zugehörige falsch negative Muster ziehen dieselben Gewichte um denselben Betrag in entgegengesetzte Richtungen, sodass sich die Änderungen über einen Durchgang genau kompensieren. Das Netz sitzt damit in einem stabilen Gleichgewicht. Eine feinere Zeitauflösung verschiebt die prinzipielle Grenze von 13 von 16 nicht, beseitigt aber Artefakte, bei denen ein zu grobes Gitter einzelne Muster falsch einordnet.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Fehler ueber die Durchgaenge
ax1.plot(fehler_verlauf, marker="o")
ax1.set_xlabel("Durchgang (Epoche)")
ax1.set_ylabel("Anzahl Fehler")
ax1.set_title("Fehler ueber die Durchgaenge")
ax1.grid(alpha=0.3)

# Entwicklung der vier Gewichte
for i in range(4):
    ax2.plot(gewichte_verlauf[:, i], label=f"w{i+1}")
ax2.set_xlabel("Durchgang (Epoche)")
ax2.set_ylabel("Gewicht")
ax2.set_title("Entwicklung der Gewichte")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Robustheit (3e): mehrmals mit verschiedenen seeds trainieren und vergleichen.
# Achtung: dauert entsprechend laenger.
# for s in range(3):
#     w_s, f_s, _ = train(patterns, targets, learning_rate=0.2, epochs=25, seed=s)
#     print(f"seed {s}: Endfehler {f_s[-1]}, Gewichte {np.round(w_s,2)}")

# Aufgabe 4: Keras: Schachbretterkennung

## 4a) Datenpräparation (Feature/Target)

Es werden alle möglichen 16 Felder des 2x2 Schachbrettmusters in Form einer 2x2 Matrix generiert, 14 falsche und 2 richtige. Es werden zusätzlich 12 richtige Duplikate hinzugefügt, um das den Datensatz zu auszugleichen ("Balancing").

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten


feature_train = np.array([
    [[0, 0], [0, 0]], #1
    [[0, 0], [0, 1]], #2
    [[0, 0], [1, 0]], #3
    [[0, 0], [1, 1]], #4
    [[0, 1], [0, 0]], #5
    [[0, 1], [0, 1]], #6
    [[0, 1], [1, 0]], #7
    [[0, 1], [1, 1]], #8
    [[1, 0], [0, 0]], #9
    [[1, 0], [0, 1]], #10
    [[1, 0], [1, 0]], #11
    [[1, 0], [1, 1]], #12
    [[1, 1], [0, 0]], #13
    [[1, 1], [0, 1]], #14
    [[1, 1], [1, 0]], #15
    [[1, 1], [1, 1]], #16
    # Ab hier die Duplikationen des richtigen Schachbrettmusters
    [[1, 0], [0, 1]], #17
    [[1, 0], [0, 1]], #18
    [[1, 0], [0, 1]], #19
    [[1, 0], [0, 1]], #20
    [[1, 0], [0, 1]], #21
    [[1, 0], [0, 1]], #22
    [[1, 0], [0, 1]], #23
    [[1, 0], [0, 1]], #24
    [[1, 0], [0, 1]], #25
    [[1, 0], [0, 1]], #26
    [[1, 0], [0, 1]], #27
    [[1, 0], [0, 1]], #28
    [[1, 0], [0, 1]], #29
    [[1, 0], [0, 1]], #30
])

target_train = np.array([
    [0], #1
    [0], #2
    [0], #3
    [0], #4
    [0], #5
    [0], #6
    [0], #7
    [0], #8
    [0], #9
    [1], #10
    [0], #11
    [0], #12
    [0], #13
    [0], #14
    [0], #15
    [0], #16
    # Ab hier die Duplikationen des richtigen Schachbrettmusters
    [1], #17
    [1], #18
    [1], #19
    [1], #20
    [1], #21
    [1], #22
    [1], #23
    [1], #24
    [1], #25
    [1], #26
    [1], #27
    [1], #28
    [1], #29
    [1],  #30
])

def randomisieren(feature_train, target_train):
    random = np.arange(len(feature_train))
    np.random.shuffle(random)
    feature_train = feature_train[random]
    target_train = target_train[random]
    return feature_train, target_train

feature_train, target_train = randomisieren(feature_train, target_train)
print("erstes Feature: ", feature_train[0]) # Test ob wirklich randomisiert wurde

## 4b) Modellaufbau (Sequential, Dense)

Es wird ein Neuronales Netz definiert. Als Output-Funktion wird "Sigmoid" verwendet, welche Werte $x \geq 0.5$ auf $1$ und $x \leq 0.5$ auf $0$ rundet.

In [ ]:
model = Sequential()
model.add(Flatten(input_shape=(2, 2)))
model.add(Dense(1, activation="sigmoid")) # Sigmoid als Output

## 4c) Kompilierung (Optimizer, Loss)

In [ ]:
model.compile(optimizer="sgd", loss="binary_crossentropy", metrics=["accuracy"])

## 4d) Training & Evaluation

Das Modell wird nun trainiert und anschließend evaluiert.

In [ ]:
model.fit(feature_train, target_train, epochs=100, verbose=0, batch_size=1)

feature_test = np.array([
    [[0, 0], [0, 0]], #1
    [[0, 0], [0, 1]], #2
    [[0, 0], [1, 0]], #3
    [[0, 0], [1, 1]], #4
    [[0, 1], [0, 0]], #5
    [[0, 1], [0, 1]], #6
    [[0, 1], [1, 0]], #7
    [[0, 1], [1, 1]], #8
    [[1, 0], [0, 0]], #9
    [[1, 0], [0, 1]], #10
    [[1, 0], [1, 0]], #11
    [[1, 0], [1, 1]], #12
    [[1, 1], [0, 0]], #13
    [[1, 1], [0, 1]], #14
    [[1, 1], [1, 0]], #15
    [[1, 1], [1, 1]]  #16
])

target_test = np.array([
    [0], #1
    [0], #2
    [0], #3
    [0], #4
    [0], #5
    [0], #6
    [0], #7
    [0], #8
    [0], #9
    [1], #10
    [0], #11
    [0], #12
    [0], #13
    [0], #14
    [0], #15
    [0] #16
])

print("erstes Feature: ", feature_train[0])
print("")
print("")
print("")
print("Testergebnisse:")
model.evaluate(feature_test, target_test)
predictions = model.predict(feature_test)
print("\nEinzelprüfung der 16 Muster:")
for i in range(len(feature_test)):

    if predictions[i] > 0.5:
        prediction_binary = 1
    else:
        prediction_binary = 0

    if prediction_binary == target_test[i]:
        test = "Richtig"
    else:
        test = "Falsch"
    
    print(f"Schachbrettmuster {i+1}: Bestimmt auf {prediction_binary}, Lösung {target_test[i]} = {test}")

Es zeigt sich nach mehrmaligen Durchläufen bei 200 Epochen, dass die Accuracy überwiegend bei 0.8 bis 1.0 liegt.

Das Randomisieren der Reihenfolge des Datensatzes beim Trainieren ändert die Konsistenz der Ergebnisse nicht signifikant.

Bei einem Durchlauf mit 10000 Epochen ist die Accuracy = 1.000 und die Loss-Funktion = 0.023. Mit diesem Training erkennt das Neuronale Netzwerk auch Floats richtig; beispielsweise 0.8 und 0.9 statt 1, sowie 0.1 und 0.2 als 0.

Außerdem ließen sich folgende Dinge beobachten:
Nach einer Korrektur des Balancing, bei der die Anzahl des richtigen Schachbrettmusters erhöht wurde, verschlechterte sich die Accuracy deutlich. Obowhl die Verteilung des richtigen zu falschen Schachbrettmuster auf 50/50 statt 30/70 gesetzt wurde, wurde die Accuracy inkonsistent. Die Zusammensetzung des Testdatensatzes wurde hierbei nicht geändert.
Die Reduzierung von "batch_size" auf 1 (statt standardmäßig 32), verbesserte das Ergebnis deutlich. Die Accuracy erhöhte sich auf die konsistenten 0.8 bis 1.0, vorher war sie bei 0.18 bis 0.85. Dies liegt daran, dass die Gewichtung nach jedem Datenpunkt pro Epoche, statt nach jeder Epoche, angepasst wurde.

# Aufgabe 5: MNIST / fashion-MNIST

## 5a) Daten importieren und präparieren

Wir importieren den Datensatz mit den handgeschriebenen Ziffern. Die Daten sind in einer 28x28 Matrix mit einem Wert von 0 bis 255.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.datasets import mnist  # oder fashion_mnist
import seaborn as sns
from sklearn.metrics import confusion_matrix



(feature_train, labels_train), (feature_test, labels_test) = mnist.load_data() # Laden der Daten

feature_train = feature_train[:10000] # Nur die ersten 10000 Bilder
labels_train = labels_train[:10000]
feature_test = feature_test[:10000]
labels_test = labels_test[:10000]

feature_train = feature_train / 255.0 # Damit die Pixelwerte zwischen 0 und 1 liegen
feature_test = feature_test / 255.0

#index = np.random.randint(0, len(feature_train))
#plt.imshow(feature_train[index], cmap='gray')
#plt.title(f"Label: {labels_train[index]}")
#plt.show()

## 5b) Netz mit nur einem Output-Layer

Es wird ein Neuronales Netz mit einem Output-Layer generiert.

In [ ]:
NN = Sequential()
NN.add(Flatten(input_shape=(28, 28)))
NN.add(Dense(10, activation="softmax")) #10 Neuronen
#NN.summary()

## 5c) Kompilieren, trainieren, evaluieren

Dieses Neuronale Netz wird nun trainiert. Hierfür wird das Netz für je 5 Epochen trainiert und der Mittelwert der Accuracy über 10 Durchläufe gemittelt. Mit diesem Ansatz soll die statistische Schwankungen, welche durch die zufällig gewählten Startvariablen verursacht wird, reduziert werden.

Parameter: 5 Epochen, 10 Durchläufe, 10000 Trainings- & Testdatenpunkte
Mittelwert (der val_accuracy): (0.9083+0.9133+0.9158+0.9164+0.9137+0.9155+0.9142+0.9144+0.9132+0.9144+0.9129)/10 = 0.91366

In [ ]:
NN.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
NN.fit(feature_train, labels_train, epochs=5, validation_data=(feature_test, labels_test))

## 5d) Confusion Matrix

Es wird eine Confusion Matrix erzeugt, welche die verwechselten Zahlen in einem anschaulichen Bild sichtbar macht.

In [ ]:
feature_test_predictions_probability = NN.predict(feature_test) # Wahrscheinlichkeiten für jede Klasse
labels_test_predictions = np.argmax(feature_test_predictions_probability, axis=1) # Höchste Wahrscheinlichkeit

confmatrix = confusion_matrix(labels_test, labels_test_predictions)

plt.figure(figsize=(10, 8)) 
sns.heatmap(confmatrix, annot=True, fmt="d", cmap="Blues") # annot = Zahlen in den Zellen, fmt = Formatierung der Zahlen, cmap = Farbskala
plt.xlabel("Vorhergesagte Klasse") # Achse
plt.ylabel("Wahre Klasse") # Achse
plt.title("Confusion Matrix") # Titel
plt.show()

Die höchsten Werte der falsch erkannten Zahlen betragen je nach Durchlauf ungefähr 50 bis 60.

## 5e) Hidden-Layer ergänzen und vergleichen

Das Neuronale Netz wird mit einem Hidden-Layer erweitert und anschließend mit einer unterschiedlichen Anzahl an Neuronen getestet. Es werden die gleichen Parameter, wie beim Trainieren des Neuronales Netzes ohne Hidden-Layer verwendet; Mittelung über 5 Epochen, 10 Durchläufe, 10000 Datenpunkte.

In [ ]:
def NN2(Anzahl):
    NN = Sequential()
    NN.add(Flatten(input_shape=(28, 28)))
    NN.add(Dense(Anzahl, activation="relu")) # Hidden-Layer
    NN.add(Dense(10, activation="softmax")) # 10 Neuronen im letzten Layer
    NN.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    NN.fit(feature_train, labels_train, epochs=5, validation_data=(feature_test, labels_test), verbose=1)
    accuracy = NN.evaluate(feature_test, labels_test)
    return accuracy[1], NN

ergebnisse = []
for i in range(10):
    print(f"Simulation {i+1}/10 läuft...")
    accuracy, NN3 = NN2(100) # Anzahl der Neuronen im Hidden-Layer
    ergebnisse.append(accuracy)

gesamt_mittelwert = np.mean(ergebnisse)

print(f"\nDurchschnittliche Test-Accuracy aus 10 Durchläufen: {gesamt_mittelwert:.4f}") # Ausgabe der gemittelten Accuracy


# Der Durchlauf über die verschiedenen Anzahlen von Neuronen im Hidden-Layer lässt sich hiermit automatisieren
#neuronen_liste = [10, 20, 50, 100, 250, 500, 1000] 
#tabelle = {}
#for anzahl in neuronen_liste:
#    ergebnisse = []
#    for i in range(10):
#        accuracy, NN3 = NN2(anzahl)
#        ergebnisse.append(accuracy)
#    tabelle[anzahl] = np.mean(ergebnisse)
#    print(f"Neuronen: {anzahl}, gemittelte Accuracy: {tabelle[anzahl]:.4f}")

Die Testergebnisse lassen sich aus folgender Tabelle entnehmen:

Parameter: 5 Epochen, 10 Durchläufe, 10000 Trainings- & Testdatenpunkte

Anzahl Neuronen

| Anzahl der Neuronen | Gemittelte Accuracy | Delta zum Vorherigem |
| :--- | :--- | :--- |
| 10 | 0.9028 | |
| 20 | 0.9206 | +0.0178, +1.93%  |
| 50 | 0.9310 | +0.0104, +1.12% |
| 100 | 0.9419 | +0.0109, +1.16% |
| 250 | 0.9489 | +0.007, +0.74% |
| 500 | 0.9533 | +0.0044, +0.46% |
| 1000 | 0.9565 | +0.0032, +0.33% |

Zu erkennen ist, dass die Erhöhung der Anzahl der Neuronen die gemittelte Accuracy verbessert. Dieser Zusammenhang ist jedoch nicht proportional; die Verbresserung der gemittelten Accuracy nimmt mit jeder Verdopplung der Anzahl der Neuronen weiter ab. 
Dies lässt sich mit folgendem Graphen visualisieren:

In [ ]:
neuronen = [10, 20, 50, 100, 250, 500, 1000]
accuracy = [0.9028, 0.9206, 0.9310, 0.9419, 0.9489, 0.9533, 0.9565]

plt.figure(figsize=(10, 6))

plt.plot(neuronen, accuracy, marker="o", alpha = 0.8)

for x, y in zip(neuronen, accuracy):
    plt.text(x, y+0.003, f"{y:.4f}", ha="center",)

plt.xlabel("Anzahl der Neuronen im Hidden-Layer")
plt.ylabel("Gemittelte Accuracy")
plt.title("Einfluss der Neuronenanzahl auf die gemittelte Accuracy")

#plt.xscale('log')
plt.xticks(neuronen, labels=["10", "\n20", "50", "100", "250", "500", "1000"])

plt.xlim(0, 1050)
plt.ylim(bottom=0.85, top=1)

plt.grid(True, linestyle='--', alpha=0.6)

plt.axhline(y=0.91366, color='r', linestyle='--', linewidth=2, label='Mittelwert ohne Hidden-Layer (0.91366)')
plt.legend(shadow=True)

plt.show()

Zu beobachten ist, dass ein Neuronales Netz mit einer zusätzlichen Hidden-Layer mit 10 Neuronen schlechter abschneidet, als ein Neuronales Netz mit nur einer direkten Layer mit 10 Neuronen. Dies ändert sich bei der Erhöhung der Epochen.

Nun wird die Confusion Matrix für das Modell mit der zusätzlichen Hidden-Layer ausgegeben. Dies wurde für jede simulierte Anzahl an Neuronen im Hidden-Layer manuell durchgeführt.

In [ ]:
feature_test_predictions_probability = NN3.predict(feature_test) 
labels_test_predictions = np.argmax(feature_test_predictions_probability, axis=1) 

confmatrix = confusion_matrix(labels_test, labels_test_predictions)

plt.figure(figsize=(10, 8)) 
sns.heatmap(confmatrix, annot=True, fmt='d', cmap='Blues') 
plt.xlabel("Vorhergesagte Klasse") 
plt.ylabel("Wahre Klasse") 
plt.title("Confusion Matrix für X Neuronen") 
plt.show()

Alle Confusion Matrizen lassen sich wie folgt ausgeben (vorher generiert und gespeichert):

In [ ]:
from IPython.display import Image, display
display(Image(filename="images/10.png"))
display(Image(filename="images/20.png"))
display(Image(filename="images/50.png"))
display(Image(filename="images/100.png"))
display(Image(filename="images/250.png"))
display(Image(filename="images/500.png"))
display(Image(filename="images/1000.png"))


Zu erkennen ist, dass die Werte der falsch erkannten Zahlen nehmen mit der Erhöhung der Anzahl der Neuronen im Hidden-Layer ab. Die Anzahl der falsch erkannten Zahlen hat ab 100 Neuronen keine Werte mehr über 50.

## Fazit

In diesem Projekt wurden verschiedene Ansätze als Aufbau und Simulation von Neuronalen Netzwerken getestet.

Im ersten Teil ... Aufgabe 2

Daurauffolgend ... Aufgabe 3

Im Anschluss wurde die selbe Herangehensweise mit der Keras-Bibliothek auf ein Schachbrettmuster durchgeführt. Dabei zeigte sich, welche Auswirkungen unterschiedliche Datenpräparationen, Balancing der Trainingsdaten, Epochenzahlen als auch die Batch-Größen auf die Stabilität und Zuverlässigkeit der Testergebnisse haben können. 

Abschließend wurde das mit der Keras-Bibliothek erstellte Neuronale Netz ausgebaut und an einem komplexeren MNIST-Schriftzeichen-Datensatz angewendet. Der Vergleich zwischen einem Neuronalen Netz ohne und mit Hidden-Layer zeigt, dass mit zunehmender Anzahl an Neuronen die Erkennungsgenauigkeit ansteigt, wobei die Verbesserung mit jeder Verdopplung der Neuronenanzahl abnahm. Dies verdeutlichen die zugehörigen Confusion-Matrizen, die zeigen, dass mit zunehmender Anzahl an Neuronen die Verwechslungen der handgeschriebenen Zahlen abnehmen.